In [ ]:
!pip install faiss-cpu scikit-learn pandas numpy joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 6.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import joblib
import faiss
from sklearn.preprocessing import StandardScaler
from pathlib import Path

In [ ]:
def fit_and_save_scaler(X: np.ndarray, scaler_path: str):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    joblib.dump(scaler, scaler_path)
    print(f"Scaler saved → {scaler_path}")
    return X_scaled

In [ ]:
def build_and_save_faiss_index(
    X_scaled: np.ndarray,
    index_path: str,
    hnsw_m: int = 32
):
    dim = X_scaled.shape[1]

    index = faiss.IndexHNSWFlat(dim, hnsw_m)
    index.hnsw.efConstruction = 200
    index.hnsw.efSearch = 50

    index.add(X_scaled)
    faiss.write_index(index, index_path)

    print(f"FAISS index saved → {index_path}")
    print(f"Total vectors indexed → {index.ntotal}")

In [ ]:
Path("artifacts/faiss").mkdir(parents=True, exist_ok=True)

In [ ]:
ENDPOINT1_FEATURES = [
    'Mg', 'Al', 'Si', 'P', 'K FC', 'Ca FC', 'Ti FC',
    'Mn FC', 'Fe FC', 'Cu FC', 'Zn FC',
    'R', 'G', 'B']

In [ ]:
df1 = pd.read_csv("/content/fallback_dataset_endpoint1.csv")

X1 = df1[ENDPOINT1_FEATURES].values.astype("float32")

In [ ]:
X1_scaled = fit_and_save_scaler(
    X1,
    "artifacts/faiss/endpoint1_scaler.joblib"
)

build_and_save_faiss_index(
    X1_scaled,
    "artifacts/faiss/endpoint1_index.faiss"
)

Scaler saved → artifacts/faiss/endpoint1_scaler.joblib
FAISS index saved → artifacts/faiss/endpoint1_index.faiss
Total vectors indexed → 31575


In [ ]:
ENDPOINT2_FEATURES = [
    'R', 'G', 'B',
    'Lat', 'Lon', 'Elevation-1', 'Elevation-2',
    'Band 1', 'Band 2', 'Band 3', 'Band 4',
    'Band 5', 'Band 6', 'Band 7', 'Band 8'
]

In [ ]:
df2 = pd.read_csv("/content/fallback_dataset_endpoint2.csv")

X2 = df2[ENDPOINT2_FEATURES].values.astype("float32")

In [ ]:
X2_scaled = fit_and_save_scaler(
    X2,
    "artifacts/faiss/endpoint2_scaler.joblib"
)

build_and_save_faiss_index(
    X2_scaled,
    "artifacts/faiss/endpoint2_index.faiss"
)

Scaler saved → artifacts/faiss/endpoint2_scaler.joblib
FAISS index saved → artifacts/faiss/endpoint2_index.faiss
Total vectors indexed → 31575


In [ ]:
idx1 = faiss.read_index("artifacts/faiss/endpoint1_index.faiss")
#idx2 = faiss.read_index("artifacts/faiss/endpoint2_index.faiss")

print("Endpoint-1 vectors:", idx1.ntotal)
#print("Endpoint-2 vectors:", idx2.ntotal)

Endpoint-1 vectors: 31575
